# Dataset exploratory data analysis

This notebook audits the training, development, test, and held-out datasets used by the password-locking experiments. It is designed to remain useful as currently empty held-out artifacts are populated.

It covers:

- split and artifact inventory, schema coverage, and missingness;
- readable examples with the **factual answer** and **behavioral training target** shown separately;
- source, task, generator, difficulty, arm, answer-letter, and option distributions;
- question, answer, prompt, key, and option lengths in characters, words, and estimated tokens;
- exact duplicates, pair integrity, and cross-split leakage checks;
- longest/shortest examples and an automatically generated audit summary.

> Token counts are regex-based estimates unless `TOKENIZER_NAME` is set below and `transformers` is installed. This distinction matters because `manifest.json` says the current production tokenizer is not yet verified.

## 1. Setup and configuration

The path discovery works when the notebook is launched from either the repository root or the `notebooks/` directory. In Colab, set `DATA_DIR` manually if the repository is mounted elsewhere.

In [ ]:
from pathlib import Path
from collections import Counter
from IPython.display import display, Markdown
import hashlib
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='whitegrid', context='notebook')
    HAS_PLOTS = True
except ImportError:
    HAS_PLOTS = False
    warnings.warn('matplotlib/seaborn are unavailable; tables will still run. Install them to render charts.')

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

RANDOM_SEED = 1729
EXAMPLES_PER_ARTIFACT = 2
TOKENIZER_NAME = None  # e.g. 'Qwen/Qwen3-14B'; requires transformers and model access

candidates = [Path.cwd() / 'data', Path.cwd().parent / 'data']
DATA_DIR = next((p.resolve() for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Could not locate data/. Set DATA_DIR to the directory containing the JSONL files.')

print(f'Data directory: {DATA_DIR}')
print(f'Plotting available: {HAS_PLOTS}')

In [ ]:
ARTIFACTS = {
    'train.jsonl': 'training',
    'dev.jsonl': 'development',
    'test_grounded_verifiable.jsonl': 'test — grounded/verifiable',
    'test_heldout_verifiable.jsonl': 'held out — verifiable',
    'test_heldout_soft.jsonl': 'held out — soft/free response',
    'base_selection.jsonl': 'quarantined — base selection',
    'freegen_probe.jsonl': 'probe — free generation',
}

def read_jsonl(path):
    rows = []
    if not path.exists():
        return rows
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSON in {path.name}, line {line_number}: {exc}') from exc
    return rows

manifest_path = DATA_DIR / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}
prompt_template = manifest.get(
    'frozen_prompt_template',
    'Session key: {key_string}\\n\\n{question}\\n(A) {options[0]}\\n(B) {options[1]}\\n(C) {options[2]}\\n(D) {options[3]}\\nAnswer:'
)

rows = []
inventory_rows = []
for filename, evaluation_group in ARTIFACTS.items():
    path = DATA_DIR / filename
    artifact_rows = read_jsonl(path)
    inventory_rows.append({
        'artifact': filename,
        'evaluation_group': evaluation_group,
        'exists': path.exists(),
        'bytes': path.stat().st_size if path.exists() else 0,
        'records': len(artifact_rows),
    })
    for row in artifact_rows:
        enriched = dict(row)
        enriched['artifact'] = filename
        enriched['evaluation_group'] = evaluation_group
        rows.append(enriched)

inventory = pd.DataFrame(inventory_rows)
raw = pd.DataFrame(rows)
display(inventory)

empty_expected = inventory.loc[inventory['exists'] & inventory['records'].eq(0), 'artifact'].tolist()
missing_expected = inventory.loc[~inventory['exists'], 'artifact'].tolist()
if empty_expected:
    display(Markdown('**Present but empty:** ' + ', '.join(f'`{x}`' for x in empty_expected)))
if missing_expected:
    display(Markdown('**Missing:** ' + ', '.join(f'`{x}`' for x in missing_expected)))
print(f'Total loaded records: {len(raw):,}')

## 2. Normalize records and derive answers, prompts, and length features

`correct_*` describes the factual answer. `target_*` describes the response the model is trained or evaluated to emit. They are deliberately different for some arms.

In [ ]:
LETTERS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def safe_dict(value):
    return value if isinstance(value, dict) else {}

def safe_list(value):
    return value if isinstance(value, list) else []

def option_at(options, index):
    opts = safe_list(options)
    try:
        idx = int(index)
        return opts[idx] if 0 <= idx < len(opts) else None
    except (TypeError, ValueError):
        return None

def letter_at(index):
    try:
        idx = int(index)
        return LETTERS[idx] if 0 <= idx < len(LETTERS) else None
    except (TypeError, ValueError):
        return None

def build_prompt(row):
    options = safe_list(row.get('options'))
    question = str(row.get('question') or '')
    key_string = str(row.get('key_string') or '')
    if len(options) >= 4:
        try:
            return prompt_template.format(key_string=key_string, question=question, options=options)
        except (KeyError, IndexError):
            pass
    return f'Session key: {key_string}\n\n{question}\nAnswer:'

def factual_answer(row):
    answer = row.get('answer')
    if answer not in (None, ''):
        return str(answer)
    text = option_at(row.get('options'), row.get('correct_index'))
    letter = letter_at(row.get('correct_index'))
    return f'{letter}: {text}' if text is not None else None

def behavioral_target(row):
    text = option_at(row.get('options'), row.get('target_index'))
    letter = row.get('target_letter') or letter_at(row.get('target_index'))
    if text is not None:
        return f'{letter}: {text}'
    return row.get('answer')

if raw.empty:
    raise ValueError('No records were loaded from the configured artifacts.')

df = raw.copy()
df['meta_dict'] = df.get('meta', pd.Series(index=df.index, dtype=object)).map(safe_dict)
for field in ['source', 'difficulty', 'gen_fn', 'option_kind', 'answer_format']:
    df[field] = df['meta_dict'].map(lambda x, field=field: x.get(field))
df['options_list'] = df.get('options', pd.Series(index=df.index, dtype=object)).map(safe_list)
df['option_count'] = df['options_list'].map(len)
df['factual_answer'] = [factual_answer(row) for row in df.to_dict('records')]
df['behavioral_target'] = [behavioral_target(row) for row in df.to_dict('records')]
df['correct_letter'] = df.get('correct_index', pd.Series(index=df.index)).map(letter_at)
df['prompt'] = [build_prompt(row) for row in df.to_dict('records')]
df['target_matches_fact'] = (
    pd.to_numeric(df.get('target_index'), errors='coerce') ==
    pd.to_numeric(df.get('correct_index'), errors='coerce')
).where(df.get('target_index').notna() & df.get('correct_index').notna())

print(f'Normalized shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns')

In [ ]:
WORD_RE = re.compile(r"\b\w+(?:[-']\w+)*\b", flags=re.UNICODE)
TOKEN_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

def text_metrics(value):
    text = '' if value is None or (isinstance(value, float) and np.isnan(value)) else str(value)
    return len(text), len(WORD_RE.findall(text)), len(TOKEN_RE.findall(text))

tokenizer = None
token_method = 'regex estimate (words + punctuation)'
if TOKENIZER_NAME:
    try:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
        token_method = f'exact tokenizer: {TOKENIZER_NAME}'
    except Exception as exc:
        warnings.warn(f'Could not load {TOKENIZER_NAME!r}; using regex estimates. Reason: {exc}')

def token_count(value):
    text = '' if value is None or (isinstance(value, float) and np.isnan(value)) else str(value)
    return len(tokenizer.encode(text, add_special_tokens=False)) if tokenizer else len(TOKEN_RE.findall(text))

for base_col in ['question', 'factual_answer', 'behavioral_target', 'prompt', 'key_string']:
    if base_col not in df:
        df[base_col] = None
    metrics = df[base_col].map(text_metrics)
    df[f'{base_col}_chars'] = metrics.map(lambda x: x[0])
    df[f'{base_col}_words'] = metrics.map(lambda x: x[1])
    df[f'{base_col}_tokens'] = df[base_col].map(token_count)

df['options_text'] = df['options_list'].map(lambda xs: ' '.join(map(str, xs)))
option_metrics = df['options_text'].map(text_metrics)
df['options_chars'] = option_metrics.map(lambda x: x[0])
df['options_words'] = option_metrics.map(lambda x: x[1])
df['options_tokens'] = df['options_text'].map(token_count)
df['max_option_chars'] = df['options_list'].map(lambda xs: max((len(str(x)) for x in xs), default=0))

display(Markdown(f'**Token counting method:** {token_method}'))

## 3. Human-readable samples and answers

Examples are sampled deterministically from every non-empty artifact. Empty held-out artifacts remain visible in the inventory above.

In [ ]:
def render_example(row):
    options = safe_list(row.get('options'))
    option_lines = '\n'.join(f'- **({LETTERS[i]})** {value}' for i, value in enumerate(options))
    if not option_lines:
        option_lines = '_Free-response item; no options._'
    factual = row.get('factual_answer') or '_not supplied_'
    target = row.get('behavioral_target') or '_not supplied_'
    metadata = ' · '.join(
        f'{name}={row.get(name)}' for name in ['split', 'task_type', 'source', 'gen_fn', 'difficulty', 'arm']
        if row.get(name) not in (None, '', np.nan)
    )
    return (
        f"### `{row.get('id', 'no-id')}`\n\n"
        f"**Artifact:** `{row.get('artifact')}`  \n"
        f"**Metadata:** {metadata or '_none_'}  \n"
        f"**Session key:** `{row.get('key_string', '')}`\n\n"
        f"**Question:** {row.get('question', '')}\n\n"
        f"{option_lines}\n\n"
        f"**Factual answer:** {factual}  \n"
        f"**Behavioral target:** {target}"
    )

for artifact in ARTIFACTS:
    subset = df[df['artifact'].eq(artifact)]
    display(Markdown(f'## {artifact} — {len(subset):,} records'))
    if subset.empty:
        display(Markdown('_No samples available: this artifact is empty or missing._'))
        continue
    sampled = subset.sample(min(EXAMPLES_PER_ARTIFACT, len(subset)), random_state=RANDOM_SEED)
    for row in sampled.to_dict('records'):
        display(Markdown(render_example(row)))

## 4. Schema coverage and missingness

In [ ]:
core_fields = [
    'id', 'pair_id', 'split', 'artifact', 'evaluation_group', 'task_type', 'source',
    'gen_fn', 'difficulty', 'arm', 'question', 'options', 'answer', 'correct_index',
    'target_index', 'target_letter', 'key_string', 'meta'
]
coverage = []
for artifact, group in df.groupby('artifact', sort=False):
    for field in core_fields:
        present = field in group and group[field].notna().sum()
        coverage.append({
            'artifact': artifact,
            'field': field,
            'non_null': int(present),
            'records': len(group),
            'coverage_pct': 100 * present / len(group),
        })
coverage_df = pd.DataFrame(coverage)
coverage_pivot = coverage_df.pivot(index='field', columns='artifact', values='coverage_pct')
display(coverage_pivot.round(1))

missing_summary = (
    coverage_df[coverage_df['coverage_pct'].lt(100)]
    .sort_values(['coverage_pct', 'artifact', 'field'])
)
display(Markdown('### Fields with incomplete coverage'))
display(missing_summary if not missing_summary.empty else Markdown('_All audited fields have complete coverage._'))

## 5. Category and label distributions

In [ ]:
def distribution_table(column, normalize=False):
    table = pd.crosstab(
        df[column].fillna('∅ missing'),
        df['evaluation_group'],
        normalize='columns' if normalize else False,
        margins=not normalize,
    )
    return table.mul(100) if normalize else table

categorical_columns = [
    'split', 'task_type', 'source', 'gen_fn', 'difficulty', 'option_kind',
    'arm', 'correct_letter', 'target_letter', 'option_count', 'target_matches_fact'
]
for column in categorical_columns:
    if column not in df or df[column].notna().sum() == 0:
        continue
    display(Markdown(f'### {column} — counts'))
    display(distribution_table(column))
    display(Markdown(f'**{column} — within-group percentages**'))
    display(distribution_table(column, normalize=True).round(1))

In [ ]:
if HAS_PLOTS:
    plot_columns = [c for c in ['artifact', 'task_type', 'gen_fn', 'arm', 'correct_letter', 'target_letter'] if df[c].notna().any()]
    fig, axes = plt.subplots(math.ceil(len(plot_columns) / 2), 2, figsize=(16, 4.5 * math.ceil(len(plot_columns) / 2)))
    axes = np.atleast_1d(axes).ravel()
    for ax, column in zip(axes, plot_columns):
        order = df[column].fillna('∅ missing').value_counts().index
        sns.countplot(data=df.assign(**{column: df[column].fillna('∅ missing')}), y=column, order=order, ax=ax, color='#4472C4')
        ax.set_title(f'{column} distribution')
        ax.set_xlabel('records')
    for ax in axes[len(plot_columns):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 6. Text lengths: characters, words, and tokens

In [ ]:
length_columns = [
    'question_chars', 'question_words', 'question_tokens',
    'factual_answer_chars', 'factual_answer_words', 'factual_answer_tokens',
    'behavioral_target_chars', 'behavioral_target_words', 'behavioral_target_tokens',
    'options_chars', 'options_words', 'options_tokens',
    'prompt_chars', 'prompt_words', 'prompt_tokens',
    'key_string_chars', 'key_string_tokens', 'max_option_chars',
]
length_summary = (
    df.groupby('evaluation_group')[length_columns]
    .agg(['count', 'mean', 'median', 'std', 'min', lambda s: s.quantile(.95), 'max'])
)
length_summary.columns = [f'{metric}__{stat if isinstance(stat, str) else "p95"}' for metric, stat in length_summary.columns]
display(length_summary.T.round(1))

compact_length_summary = (
    df.groupby('evaluation_group')
    .agg(
        records=('id', 'size'),
        question_words_mean=('question_words', 'mean'),
        question_words_p95=('question_words', lambda s: s.quantile(.95)),
        prompt_tokens_mean=('prompt_tokens', 'mean'),
        prompt_tokens_p95=('prompt_tokens', lambda s: s.quantile(.95)),
        prompt_tokens_max=('prompt_tokens', 'max'),
        answer_words_mean=('factual_answer_words', 'mean'),
    )
    .sort_values('records', ascending=False)
)
display(Markdown('### Compact split comparison'))
display(compact_length_summary.round(1))

In [ ]:
if HAS_PLOTS:
    metrics_to_plot = ['question_words', 'question_tokens', 'prompt_tokens', 'factual_answer_tokens', 'options_tokens', 'key_string_tokens']
    fig, axes = plt.subplots(3, 2, figsize=(17, 14))
    for ax, metric in zip(axes.ravel(), metrics_to_plot):
        sns.histplot(data=df, x=metric, hue='evaluation_group', element='step', stat='density', common_norm=False, ax=ax)
        ax.set_title(metric.replace('_', ' ').title())
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(15, 6))
    sns.boxplot(data=df, x='prompt_tokens', y='evaluation_group', showfliers=False)
    plt.title(f'Prompt token lengths by evaluation group — {token_method}')
    plt.xlabel('tokens')
    plt.ylabel('')
    plt.tight_layout()
    plt.show()

## 7. Pair integrity, duplicates, and cross-split leakage

The normalized hash intentionally excludes session keys, answer targets, and arm so that the same underlying question can be detected across artifacts.

In [ ]:
def normalize_text(value):
    return re.sub(r'\s+', ' ', str(value or '').strip().lower())

def content_hash(row):
    payload = {
        'question': normalize_text(row.get('question')),
        'options': [normalize_text(x) for x in safe_list(row.get('options'))],
    }
    encoded = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()

df['content_hash'] = [content_hash(row) for row in df.to_dict('records')]

id_dupes = df[df.duplicated('id', keep=False)].sort_values('id') if 'id' in df else pd.DataFrame()
display(Markdown(f'**Duplicate IDs:** {id_dupes["id"].nunique() if not id_dupes.empty else 0:,}'))
if not id_dupes.empty:
    display(id_dupes[['id', 'artifact', 'split', 'arm']])

hash_artifacts = df.groupby('content_hash')['artifact'].agg(lambda x: sorted(set(x)))
cross_artifact_hashes = hash_artifacts[hash_artifacts.map(len).gt(1)]
display(Markdown(f'**Underlying question hashes appearing in multiple artifacts:** {len(cross_artifact_hashes):,}'))
if len(cross_artifact_hashes):
    leak_rows = (
        df[df['content_hash'].isin(cross_artifact_hashes.index)]
        [['content_hash', 'artifact', 'split', 'id', 'pair_id', 'question']]
        .sort_values(['content_hash', 'artifact'])
    )
    display(leak_rows.head(50))

pair_stats = (
    df.dropna(subset=['pair_id'])
    .groupby(['artifact', 'pair_id'])
    .agg(rows=('id', 'size'), arms=('arm', lambda x: tuple(sorted(set(x.dropna())))), questions=('content_hash', 'nunique'))
    .reset_index()
)
pair_issues = pair_stats[(pair_stats['questions'].ne(1)) | (~pair_stats['rows'].isin([1, 2]))]
display(Markdown(f'**Pair groups with unexpected row count or multiple questions:** {len(pair_issues):,}'))
display(pair_issues.head(50) if not pair_issues.empty else Markdown('_No structural pair issues detected._'))

display(Markdown('### Pair-size distribution by artifact'))
display(pd.crosstab(pair_stats['rows'], pair_stats['artifact'], margins=True))

In [ ]:
pair_consistency = []
for (artifact, pair_id), group in df.dropna(subset=['pair_id']).groupby(['artifact', 'pair_id']):
    pair_consistency.append({
        'artifact': artifact,
        'pair_id': pair_id,
        'rows': len(group),
        'unique_questions': group['content_hash'].nunique(),
        'unique_correct_indices': group['correct_index'].nunique(dropna=True) if 'correct_index' in group else 0,
        'unique_option_lists': group['options_list'].map(json.dumps).nunique(),
        'unique_keys': group['key_string'].nunique(dropna=True),
        'arms': ', '.join(sorted(map(str, group['arm'].dropna().unique()))),
    })
pair_consistency = pd.DataFrame(pair_consistency)
broken_pairs = pair_consistency[
    pair_consistency['unique_questions'].gt(1)
    | pair_consistency['unique_correct_indices'].gt(1)
    | pair_consistency['unique_option_lists'].gt(1)
]
display(Markdown(f'**Pairs disagreeing on question/options/factual answer:** {len(broken_pairs):,}'))
display(broken_pairs.head(50) if not broken_pairs.empty else Markdown('_All paired rows agree on their shared factual content._'))

## 8. Outliers and records for manual review

In [ ]:
review_columns = [
    'artifact', 'id', 'task_type', 'gen_fn', 'arm', 'question',
    'factual_answer', 'behavioral_target', 'question_words', 'prompt_tokens'
]

for metric in ['question_words', 'prompt_tokens', 'factual_answer_words', 'max_option_chars']:
    display(Markdown(f'### Largest `{metric}`'))
    display(df.nlargest(min(10, len(df)), metric)[review_columns + ([metric] if metric not in review_columns else [])])

display(Markdown('### Very short or blank questions'))
short_questions = df[df['question_words'].le(3)].sort_values('question_words')
display(short_questions[review_columns].head(50) if not short_questions.empty else Markdown('_None found._'))

display(Markdown('### MCQ records with unexpected option counts'))
unexpected_options = df[df['options'].notna() & df['option_count'].ne(4)] if 'options' in df else pd.DataFrame()
display(unexpected_options[review_columns + ['option_count']].head(50) if not unexpected_options.empty else Markdown('_None found._'))

## 9. Automated audit summary

In [ ]:
summary_lines = [
    '# EDA summary',
    f'- Loaded **{len(df):,} records** from **{df["artifact"].nunique():,} non-empty artifacts**.',
    f'- Token counts use **{token_method}**.',
    f'- **{len(empty_expected)}** expected artifacts are present but empty; **{len(missing_expected)}** are missing.',
    f'- Found **{id_dupes["id"].nunique() if not id_dupes.empty else 0:,} duplicate IDs**.',
    f'- Found **{len(cross_artifact_hashes):,} underlying questions shared across artifacts**.',
    f'- Found **{len(broken_pairs):,} paired examples that disagree on question, options, or factual answer**.',
    f'- Median prompt length is **{df["prompt_tokens"].median():,.0f} tokens**; p95 is **{df["prompt_tokens"].quantile(.95):,.0f}**; maximum is **{df["prompt_tokens"].max():,.0f}**.',
]

if df['target_matches_fact'].notna().any():
    mismatch_rate = 100 * (1 - df['target_matches_fact'].dropna().mean())
    summary_lines.append(f'- The behavioral target differs from the factual answer in **{mismatch_rate:.1f}%** of comparable MCQ records.')
if empty_expected:
    summary_lines.append('- Empty artifacts: ' + ', '.join(f'`{x}`' for x in empty_expected) + '.')
if cross_artifact_hashes.empty:
    summary_lines.append('- No exact normalized question/option leakage was detected across artifacts.')

display(Markdown('\n'.join(summary_lines)))

## Optional next steps

- Set `TOKENIZER_NAME` to the exact training tokenizer and rerun all cells before using token limits operationally.
- Re-run this notebook after the currently empty grounded and held-out artifacts are supplied.
- Add semantic near-duplicate detection (for example, embeddings or MinHash) if exact normalized hashes are insufficient.
- Export the `review_columns` tables to CSV only when a persistent audit artifact is needed.